# DCI vs agentic RAG

Compare two ways an agent reaches your corpus: ranked sections or tools that list, search, and read pages. Use an interactive coding assistant to run the same experiment loop with each interface and inspect the evidence.

## Learn | Create | Grow

### Learn
Two ways for an agent to reach documents: read the pages directly through a model-readable wiki, or call a retriever tool. One loop, two tool sets, and what each one reads.


### Create
A wiki index generated from your corpus, both modes scored over your eval cases, and a table of answer quality, calls, evidence read, and latency per mode.


### Grow
Keep the cheaper mode that passes your questions and write down what kind of question would make you switch. Tell your team one question where the modes diverged.


**Estimated time:** 35 minutes
**Reads:** corpus, eval_cases
**Writes:** wiki, agentic_runs

## Setup

Open the repository in a coding assistant with file and terminal access. Follow the messages below; the README documents the experiment tools and environment setup.

Product documentation: [Codex](https://developers.openai.com/codex/), [Claude Code](https://code.claude.com/docs/en/overview), [VS Code Copilot](https://code.visualstudio.com/docs/agents/overview).

### Message to your assistant

> Read the README beside this guide and inspect agentic_tools.py. Run inspect. Tell me which model and corpus are active, whether the cases come from workspace or seed, and the page and section counts. Do not save results yet.

You should see the model, input sources, page names, and eval cases. Stop here if the corpus or cases are empty.

### Read the implementation

Open `agentic_tools.py` to inspect the algorithms; the README lists its commands. Keep experiment results for later inspection, and rerun only when a task calls for it.

# Learn


## Task 1 of 5 — Build the wiki

An agent with file tools needs a map, or it reads pages at random. The wiki is one markdown index: every page name, what a reader should use it for, and its section headings. The skeleton is built from the headings by hand. The model writes the one-line purpose for each page, from a digest, and you correct it.

### Message to your assistant

> Run wiki and show me the proposed purpose lines beside the page headings. Explain how outline builds the skeleton before the model adds descriptions. Let me check the descriptions against the source pages before we use a reviewed wiki for comparison.

You should see one wiki row per page. Purpose lines are model proposals. Stop here if a description does not match its page.

## Task 2 of 5 — Two corpus interfaces

Agentic RAG gets one tool. `search_chunks` returns the top sections by BM25 and can be called again with a new query, but it cannot list pages or ask for a whole one. DCI gets three: `list_pages` returns the wiki, `grep_wiki` returns matching lines with the page and line number, `read_page` returns a whole page. In DCI the agent, not a retriever, decides what to read next.

### Message to your assistant

> Run interfaces for the first eval question. Show the ranked sections, matching lines, and a whole-page result. Walk me through search_chunks, list_pages, grep_wiki, and read_page in the source. Which evidence and limits does each expose?

You should see section IDs, page-and-line matches, and page text. Stop here if you cannot trace a result back to its source. Whole-page reads are capped at 12,000 characters.

### ❓ Question
Which of the four tools could leak one user's transcript into another user's answer, and what would you check on the caller before running it?

Answer:

## Task 3 of 5 — One loop for both

Both modes use the same model, question, system prompt, and turn limit. Only the tool interface changes. The trace records calls, returned evidence, and elapsed time. This controls the setup; model variability still affects individual runs.

### Message to your assistant

> Run compare on the first eval question, using the reviewed wiki if available. Show both answers and their full tool traces, evidence characters, elapsed time, and stop reason. Explain the message loop in run_agent. Keep the comparison inside that loop so both modes use the same model and settings.

You should see both answers and the evidence each received. Stop here if either reaches the turn limit or answers without evidence; inspect that failure before scoring.

<details><summary>Recorded experiment: inspect calls beside scores</summary>

Real seed run with `gpt-4.1-mini`. These are tool outputs, not an interactive UI capture. Full runs and evidence: `data/recorded_experiments.json`.

| Case | Mode | Judge /10 | Calls | Stop |
|---|---|---|---|---|
| v01 | agentic_rag | 7 | 1 | answer |
| v01 | dci | 9 | 1 | answer |
| v04 | agentic_rag | 9 | 0 | answer |
| v04 | dci | 10 | 1 | answer |

A high judge score can coexist with zero evidence calls. That violates the experiment's evidence-only instruction even when the answer sounds right. Inspect the trace before choosing a mode. A new run may differ.

</details>

# Create


## Task 4 of 5 — Score every case

Run every eval case through both modes. Your judge scores each answer against the reference from the eval case, 0 to 10. If the case names the pages that hold the evidence, the record also says whether the answer named one of them. Calls, evidence characters, and latency go in the same row, so a correct answer cannot hide a wasteful route.

### Message to your assistant

> Run score over the current eval cases with both modes, using the same reviewed wiki. Show scores and rationales per case, whether each answer names an expected page, tool calls, evidence characters, latency, and stop reason. Keep failed or missing judge scores visible. Do not save yet.

You should see two runs per case. Naming an expected page is a string check, not proof that the citation supports the answer. Stop here if a judge score is missing.

### ❓ Question
Find the case with the biggest score gap between modes. Did the losing mode retrieve the wrong evidence, or retrieve the right evidence and answer badly?

Answer:

## Task 5 of 5 — Inspect the difference

Read the biggest score gap and both routes before comparing averages. A retriever may find a useful section cheaply; file navigation may help with exact strings or evidence across pages. These are hypotheses to test on your cases. Neither interface is guaranteed to win.

### Message to your assistant

> Using the results already produced, show mean score, calls, evidence characters, and latency by mode. Open the evidence for the largest score gap. Separate retrieval mistakes from answer mistakes. Do not choose an interface for my product; I will make that decision.

You should see a two-row summary and the underlying runs. Stop here if a conclusion depends only on averages or on page-name matches. Evidence characters and latency are proxies, not token billing.

## Your turn

Add two questions from your own product: one that needs an exact string from a page, and one that needs evidence from two pages. Name the pages before you run either mode. Then run both modes and say which interface you would ship for each question and why.

Describe your two cases and expected pages to your assistant. Ask it to run those cases only after you have supplied your reasoning. You can inspect or modify the existing tools once you have chosen your approach.

<details><summary>Save results (optional)</summary>

Ask your assistant to run the tool with `--save` to write measured results through `helpers.workspace`. This runs a new experiment. Otherwise, readers use existing workspace artifacts or the labeled seed fallback.

</details>

# Grow


## From prototype to production

| What we built | Production equivalent |
|---|---|
| BM25 over markdown sections | Dense or hybrid retrieval with a reranker and index freshness |
| Every page visible to every call | Per-user authorisation on list, search, and read |
| A wiki index written once | Generated navigation with link checks and page ownership |
| One judge against a reference line | Human-calibrated scoring of answers and citations |
| In-process tools and a turn limit | Timeouts, rate limits, tracing, and durable run records |
| Two modes compared on a handful of cases | A routing decision reviewed as the corpus changes |

## Responsible controls

- File tools scoped to the corpus directory and read-only.
- Evidence read per answer logged so cost is visible per mode.
- A rule for which mode handles which question type, reviewed as the corpus grows.


## Grow further

- Swap `search_chunks` for the best rung of your retrieval ladder and rerun the comparison; note which cases change sides.
- Add a caller id to every DCI tool and refuse `read_page` on a transcript that belongs to another user.
- Write a router: send a question to DCI only when the wiki index names a page for one of its terms, otherwise to agentic RAG, and compare cost per case.